# 📊 Fase 5a: Evaluación Master de Modelos

**Objetivo:** Evaluación concisa y comparación de todos los modelos desarrollados en la Fase 4
 
**Scope:**
- Recuperar experimentos MLflow de regresión, clasificación, temporal y estratificación
- Comparar rendimiento con métricas clínicamente relevantes
- Identificar modelos ganadores para cada tarea
- Preparar base para optimización clínica
 
**Duración estimada:** 30-45 minutos

---

## Configuración inicial

In [1]:
# Configuración inicial
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import mlflow
import mlflow.sklearn
from mlflow.tracking import MlflowClient
import warnings
warnings.filterwarnings('ignore')

# Configuración de visualización
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 8)

print("🎯 FASE 5A: EVALUACIÓN MASTER DE MODELOS")
print("=" * 50)

🎯 FASE 5A: EVALUACIÓN MASTER DE MODELOS


## Configurar MLflow 

In [2]:
# Configurar MLflow
mlflow.set_tracking_uri("file:./mlruns")
client = MlflowClient()

print("📋 Conexión a MLflow establecida")
print(f"📁 Tracking URI: {mlflow.get_tracking_uri()}")


📋 Conexión a MLflow establecida
📁 Tracking URI: file:./mlruns


## 🔍 1. Recuperación de Experimentos

In [ ]:
## 🔍 1. Recuperación de Experimentos

# %%
# Recuperar todos los experimentos de la Fase 4
experiments = {
    'regression': 'alzheimer-regression-models',
    'classification': 'alzheimer-classification-models', 
    'temporal': 'alzheimer-temporal-analysis',
    'stratification': 'alzheimer-risk-stratification'
}

experiment_data = {}

for task, exp_name in experiments.items():
    try:
        experiment = mlflow.get_experiment_by_name(exp_name)
        if experiment:
            runs = mlflow.search_runs(
                experiment_ids=[experiment.experiment_id],
                filter_string="status = 'FINISHED'",
                order_by=["start_time DESC"]
            )
            experiment_data[task] = {
                'experiment_id': experiment.experiment_id,
                'runs': runs,
                'n_runs': len(runs)
            }
            print(f"✅ {task.upper()}: {len(runs)} runs encontrados")
        else:
            print(f"❌ {task.upper()}: Experimento no encontrado")
            experiment_data[task] = {'runs': pd.DataFrame(), 'n_runs': 0}
    except Exception as e:
        print(f"❌ Error recuperando {task}: {str(e)}")
        experiment_data[task] = {'runs': pd.DataFrame(), 'n_runs': 0}

total_runs = sum([data['n_runs'] for data in experiment_data.values()])
print(f"\n📊 TOTAL DE RUNS RECUPERADOS: {total_runs}")


❌ REGRESSION: Experimento no encontrado
❌ CLASSIFICATION: Experimento no encontrado
❌ TEMPORAL: Experimento no encontrado
❌ STRATIFICATION: Experimento no encontrado

📊 TOTAL DE RUNS RECUPERADOS: 0


## 📈 2. Evaluación de Modelos de Regresión

In [ ]:
## 📈 2. Evaluación de Modelos de Regresión

# %%
print("🎯 EVALUACIÓN: MODELOS DE REGRESIÓN (composite_risk_score)")
print("=" * 60)

regression_runs = experiment_data['regression']['runs']

if not regression_runs.empty:
    # Métricas clave para regresión
    regression_metrics = ['test_rmse', 'test_mae', 'test_r2', 'cv_rmse_mean', 'cv_r2_mean']
    
    # Filtrar métricas disponibles
    available_metrics = [col for col in regression_metrics if f'metrics.{col}' in regression_runs.columns]
    
    if available_metrics:
        # Crear tabla de comparación
        regression_comparison = regression_runs[['tags.model_name'] + [f'metrics.{m}' for m in available_metrics]].copy()
        regression_comparison.columns = ['Model'] + available_metrics
        regression_comparison = regression_comparison.dropna().round(4)
        
        print("📊 Top 5 Modelos de Regresión:")
        if 'test_r2' in available_metrics:
            top_regression = regression_comparison.nlargest(5, 'test_r2')
        elif 'cv_r2_mean' in available_metrics:
            top_regression = regression_comparison.nlargest(5, 'cv_r2_mean')
        else:
            top_regression = regression_comparison.nsmallest(5, available_metrics[0])
        
        print(top_regression.to_string(index=False))
        
        # Identificar modelo ganador
        if 'test_r2' in available_metrics:
            winner_idx = regression_comparison['test_r2'].idxmax()
            winner_metric = 'test_r2'
        elif 'cv_r2_mean' in available_metrics:
            winner_idx = regression_comparison['cv_r2_mean'].idxmax()
            winner_metric = 'cv_r2_mean'
        else:
            winner_idx = regression_comparison[available_metrics[0]].idxmin()
            winner_metric = available_metrics[0]
            
        regression_winner = regression_comparison.loc[winner_idx]
        print(f"\n🏆 GANADOR REGRESIÓN: {regression_winner['Model']}")
        print(f"📈 Mejor {winner_metric}: {regression_winner[winner_metric]:.4f}")
        
        # Guardar resultado
        best_regression = {
            'task': 'regression',
            'model': regression_winner['Model'],
            'metric': winner_metric,
            'value': regression_winner[winner_metric],
            'clinical_relevance': 'Predicción precisa del score de riesgo continuo'
        }
    else:
        print("❌ No se encontraron métricas de regresión")
        best_regression = None
else:
    print("❌ No hay datos de regresión disponibles")
    best_regression = None

## 🎯 3. Evaluación de Modelos de Clasificación

In [ ]:
## 🎯 3. Evaluación de Modelos de Clasificación

# %%
print("\n🎯 EVALUACIÓN: MODELOS DE CLASIFICACIÓN (risk_category)")
print("=" * 60)

classification_runs = experiment_data['classification']['runs']

if not classification_runs.empty:
    # Métricas clave para clasificación (enfoque clínico)
    classification_metrics = ['test_f1_macro', 'test_sensitivity_high', 'test_specificity_high', 
                            'test_precision_high', 'test_recall_high', 'cv_f1_macro_mean']
    
    # Filtrar métricas disponibles
    available_metrics = [col for col in classification_metrics if f'metrics.{col}' in classification_runs.columns]
    
    if available_metrics:
        # Crear tabla de comparación
        classification_comparison = classification_runs[['tags.model_name'] + [f'metrics.{m}' for m in available_metrics]].copy()
        classification_comparison.columns = ['Model'] + available_metrics
        classification_comparison = classification_comparison.dropna().round(4)
        
        print("📊 Top 5 Modelos de Clasificación:")
        # Priorizar sensibilidad para detección temprana
        if 'test_sensitivity_high' in available_metrics:
            top_classification = classification_comparison.nlargest(5, 'test_sensitivity_high')
            priority_metric = 'test_sensitivity_high'
        elif 'test_f1_macro' in available_metrics:
            top_classification = classification_comparison.nlargest(5, 'test_f1_macro')
            priority_metric = 'test_f1_macro'
        else:
            top_classification = classification_comparison.nlargest(5, available_metrics[0])
            priority_metric = available_metrics[0]
            
        print(top_classification.to_string(index=False))
        
        # Identificar modelo ganador (balance sensibilidad/especificidad)
        if 'test_sensitivity_high' in available_metrics and 'test_specificity_high' in available_metrics:
            # Score balanceado para uso clínico
            classification_comparison['clinical_score'] = (
                classification_comparison['test_sensitivity_high'] * 0.6 +  # Priorizar sensibilidad
                classification_comparison['test_specificity_high'] * 0.4
            )
            winner_idx = classification_comparison['clinical_score'].idxmax()
            winner_metric = 'clinical_score'
        else:
            winner_idx = classification_comparison[priority_metric].idxmax()
            winner_metric = priority_metric
            
        classification_winner = classification_comparison.loc[winner_idx]
        print(f"\n🏆 GANADOR CLASIFICACIÓN: {classification_winner['Model']}")
        print(f"📈 Score clínico: {classification_winner.get(winner_metric, 'N/A'):.4f}")
        
        if 'test_sensitivity_high' in available_metrics:
            print(f"🎯 Sensibilidad (High Risk): {classification_winner.get('test_sensitivity_high', 'N/A'):.4f}")
        if 'test_specificity_high' in available_metrics:
            print(f"🎯 Especificidad (High Risk): {classification_winner.get('test_specificity_high', 'N/A'):.4f}")
        
        # Guardar resultado
        best_classification = {
            'task': 'classification',
            'model': classification_winner['Model'],
            'metric': winner_metric,
            'value': classification_winner[winner_metric],
            'clinical_relevance': 'Balance sensibilidad/especificidad para detección temprana'
        }
    else:
        print("❌ No se encontraron métricas de clasificación")
        best_classification = None
else:
    print("❌ No hay datos de clasificación disponibles")
    best_classification = None

## ⏰ 4. Evaluación de Modelos Temporales

In [ ]:
## ⏰ 4. Evaluación de Modelos Temporales

# %%
print("\n🎯 EVALUACIÓN: MODELOS TEMPORALES")
print("=" * 60)

temporal_runs = experiment_data['temporal']['runs']

if not temporal_runs.empty:
    # Métricas clave para análisis temporal
    temporal_metrics = ['mae', 'rmse', 'trend_accuracy', 'prediction_stability']
    
    # Filtrar métricas disponibles
    available_metrics = [col for col in temporal_metrics if f'metrics.{col}' in temporal_runs.columns]
    
    if available_metrics:
        # Crear tabla de comparación
        temporal_comparison = temporal_runs[['tags.model_name'] + [f'metrics.{m}' for m in available_metrics]].copy()
        temporal_comparison.columns = ['Model'] + available_metrics
        temporal_comparison = temporal_comparison.dropna().round(4)
        
        print("📊 Modelos Temporales:")
        print(temporal_comparison.to_string(index=False))
        
        # Identificar modelo ganador
        if 'trend_accuracy' in available_metrics:
            winner_idx = temporal_comparison['trend_accuracy'].idxmax()
            winner_metric = 'trend_accuracy'
        elif 'mae' in available_metrics:
            winner_idx = temporal_comparison['mae'].idxmin()
            winner_metric = 'mae'
        else:
            winner_idx = temporal_comparison[available_metrics[0]].idxmin()
            winner_metric = available_metrics[0]
            
        temporal_winner = temporal_comparison.loc[winner_idx]
        print(f"\n🏆 GANADOR TEMPORAL: {temporal_winner['Model']}")
        print(f"📈 Mejor {winner_metric}: {temporal_winner[winner_metric]:.4f}")
        
        # Guardar resultado
        best_temporal = {
            'task': 'temporal',
            'model': temporal_winner['Model'],
            'metric': winner_metric,
            'value': temporal_winner[winner_metric],
            'clinical_relevance': 'Seguimiento de evolución temporal del riesgo'
        }
    else:
        print("❌ No se encontraron métricas temporales")
        best_temporal = None
else:
    print("❌ No hay datos temporales disponibles")
    best_temporal = None


## 🧬 5. Evaluación de Modelos de Estratificación

In [ ]:
## 🧬 5. Evaluación de Modelos de Estratificación

# %%
print("\n🎯 EVALUACIÓN: MODELOS DE ESTRATIFICACIÓN")
print("=" * 60)

stratification_runs = experiment_data['stratification']['runs']

if not stratification_runs.empty:
    # Métricas clave para estratificación
    stratification_metrics = ['silhouette_score', 'calinski_harabasz', 'davies_bouldin', 'n_clusters']
    
    # Filtrar métricas disponibles
    available_metrics = [col for col in stratification_metrics if f'metrics.{col}' in stratification_runs.columns]
    
    if available_metrics:
        # Crear tabla de comparación
        stratification_comparison = stratification_runs[['tags.model_name'] + [f'metrics.{m}' for m in available_metrics]].copy()
        stratification_comparison.columns = ['Model'] + available_metrics
        stratification_comparison = stratification_comparison.dropna().round(4)
        
        print("📊 Modelos de Estratificación:")
        print(stratification_comparison.to_string(index=False))
        
        # Identificar modelo ganador (silhouette score es el más interpretable)
        if 'silhouette_score' in available_metrics:
            winner_idx = stratification_comparison['silhouette_score'].idxmax()
            winner_metric = 'silhouette_score'
        elif 'calinski_harabasz' in available_metrics:
            winner_idx = stratification_comparison['calinski_harabasz'].idxmax()
            winner_metric = 'calinski_harabasz'
        else:
            winner_idx = stratification_comparison[available_metrics[0]].idxmax()
            winner_metric = available_metrics[0]
            
        stratification_winner = stratification_comparison.loc[winner_idx]
        print(f"\n🏆 GANADOR ESTRATIFICACIÓN: {stratification_winner['Model']}")
        print(f"📈 Mejor {winner_metric}: {stratification_winner[winner_metric]:.4f}")
        
        if 'n_clusters' in available_metrics:
            print(f"🎯 Número de clusters: {int(stratification_winner['n_clusters'])}")
        
        # Guardar resultado
        best_stratification = {
            'task': 'stratification',
            'model': stratification_winner['Model'],
            'metric': winner_metric,
            'value': stratification_winner[winner_metric],
            'clinical_relevance': 'Identificación de fenotipos de riesgo diferenciados'
        }
    else:
        print("❌ No se encontraron métricas de estratificación")
        best_stratification = None
else:
    print("❌ No hay datos de estratificación disponibles")
    best_stratification = None



## 🏆 6. Resumen de Modelos Ganadores

In [ ]:
## 🏆 6. Resumen de Modelos Ganadores

# %%
print("\n" + "="*70)
print("🏆 RESUMEN DE MODELOS GANADORES - FASE 4")
print("="*70)

# Compilar resultados
winners = []
for result in [best_regression, best_classification, best_temporal, best_stratification]:
    if result:
        winners.append(result)

if winners:
    # Crear DataFrame de resumen
    winners_df = pd.DataFrame(winners)
    
    print("\n📊 TABLA DE GANADORES:")
    for _, winner in winners_df.iterrows():
        print(f"\n🎯 {winner['task'].upper()}:")
        print(f"   Model: {winner['model']}")
        print(f"   Metric: {winner['metric']} = {winner['value']:.4f}")
        print(f"   Clinical: {winner['clinical_relevance']}")
    
    # Guardar resumen para siguientes notebooks
    winners_df.to_csv('../results/model_winners_summary.csv', index=False)
    print(f"\n💾 Resumen guardado en: ../results/model_winners_summary.csv")
    
else:
    print("❌ No se pudieron identificar modelos ganadores")
    winners_df = pd.DataFrame()

## 📊 7. Visualización de Rendimiento

In [ ]:
## 📊 7. Visualización de Rendimiento

# %%
if not winners_df.empty:
    print("\n📈 GENERANDO VISUALIZACIONES DE RENDIMIENTO...")
    
    # Configurar subplots
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    fig.suptitle('🏆 Rendimiento de Modelos Ganadores por Tarea', fontsize=16, fontweight='bold')
    
    # Gráfico 1: Comparación de métricas de regresión
    if best_regression and not regression_runs.empty:
        ax1 = axes[0, 0]
        reg_metrics = ['test_rmse', 'test_mae', 'test_r2']
        reg_available = [m for m in reg_metrics if f'metrics.{m}' in regression_runs.columns]
        
        if reg_available:
            reg_data = regression_runs[[f'metrics.{m}' for m in reg_available]].iloc[:5]
            reg_data.columns = reg_available
            reg_data.plot(kind='bar', ax=ax1, alpha=0.7)
            ax1.set_title('🎯 Top 5 Modelos Regresión', fontweight='bold')
            ax1.set_xlabel('Modelos')
            ax1.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
            ax1.tick_params(axis='x', rotation=45)
    
    # Gráfico 2: Comparación de métricas de clasificación
    if best_classification and not classification_runs.empty:
        ax2 = axes[0, 1]
        class_metrics = ['test_f1_macro', 'test_sensitivity_high', 'test_specificity_high']
        class_available = [m for m in class_metrics if f'metrics.{m}' in classification_runs.columns]
        
        if class_available:
            class_data = classification_runs[[f'metrics.{m}' for m in class_available]].iloc[:5]
            class_data.columns = class_available
            class_data.plot(kind='bar', ax=ax2, alpha=0.7)
            ax2.set_title('🎯 Top 5 Modelos Clasificación', fontweight='bold')
            ax2.set_xlabel('Modelos')
            ax2.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
            ax2.tick_params(axis='x', rotation=45)
    
    # Gráfico 3: Distribución de scores por tarea
    ax3 = axes[1, 0]
    if len(winners) > 1:
        winner_values = [w['value'] for w in winners]
        winner_tasks = [w['task'].title() for w in winners]
        
        bars = ax3.bar(winner_tasks, winner_values, alpha=0.7, color=['skyblue', 'lightgreen', 'orange', 'pink'])
        ax3.set_title('🏆 Scores de Modelos Ganadores', fontweight='bold')
        ax3.set_ylabel('Score Value')
        
        # Añadir valores en las barras
        for bar, value in zip(bars, winner_values):
            ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, 
                    f'{value:.3f}', ha='center', va='bottom', fontweight='bold')
    
    # Gráfico 4: Número de runs por experimento
    ax4 = axes[1, 1]
    exp_names = list(experiments.keys())
    exp_counts = [experiment_data[exp]['n_runs'] for exp in exp_names]
    
    bars = ax4.bar([name.title() for name in exp_names], exp_counts, alpha=0.7, color='lightcoral')
    ax4.set_title('📊 Número de Runs por Experimento', fontweight='bold')
    ax4.set_ylabel('Número de Runs')
    
    # Añadir valores en las barras
    for bar, count in zip(bars, exp_counts):
        ax4.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, 
                str(count), ha='center', va='bottom', fontweight='bold')
    
    plt.tight_layout()
    plt.savefig('../results/model_evaluation_summary.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print("📁 Visualización guardada en: ../results/model_evaluation_summary.png")


## 📋 8. Recomendaciones para Optimización Clínica

In [ ]:
## 📋 8. Recomendaciones para Optimización Clínica

# %%
print("\n" + "="*70)
print("📋 RECOMENDACIONES PARA OPTIMIZACIÓN CLÍNICA")
print("="*70)

recommendations = []

if best_regression:
    recommendations.append({
        'Task': 'Regresión',
        'Model': best_regression['model'],
        'Priority': 'ALTA',
        'Optimization': 'Calibrar thresholds para scores de riesgo clínico',
        'Clinical_Focus': 'Precisión en rangos de alto riesgo (>0.7)'
    })

if best_classification:
    recommendations.append({
        'Task': 'Clasificación', 
        'Model': best_classification['model'],
        'Priority': 'CRÍTICA',
        'Optimization': 'Optimizar threshold para maximizar sensibilidad',
        'Clinical_Focus': 'Detección temprana - minimizar falsos negativos'
    })

if best_temporal:
    recommendations.append({
        'Task': 'Temporal',
        'Model': best_temporal['model'], 
        'Priority': 'MEDIA',
        'Optimization': 'Validar estabilidad en ventanas temporales',
        'Clinical_Focus': 'Seguimiento longitudinal de pacientes'
    })

if best_stratification:
    recommendations.append({
        'Task': 'Estratificación',
        'Model': best_stratification['model'],
        'Priority': 'BAJA',
        'Optimization': 'Validar interpretabilidad clínica de clusters',
        'Clinical_Focus': 'Personalización de tratamiento por fenotipo'
    })

if recommendations:
    recommendations_df = pd.DataFrame(recommendations)
    print(recommendations_df.to_string(index=False))
    
    # Guardar recomendaciones
    recommendations_df.to_csv('../results/clinical_optimization_recommendations.csv', index=False)
    print(f"\n💾 Recomendaciones guardadas en: ../results/clinical_optimization_recommendations.csv")


## 🎯 9. Preparación para Fase 5B

In [ ]:
## 🎯 9. Preparación para Fase 5B

# %%
print("\n" + "="*70)
print("🎯 PREPARACIÓN PARA FASE 5B: OPTIMIZACIÓN CLÍNICA")
print("="*70)

# Generar configuración para el siguiente notebook
next_phase_config = {
    'priority_models': {},
    'clinical_thresholds': {
        'high_risk_score': 0.7,
        'moderate_risk_score': 0.4,
        'sensitivity_target': 0.85,
        'specificity_minimum': 0.75
    },
    'evaluation_focus': [
        'Calibración de probabilidades',
        'Análisis de errores por subgrupo',
        'Optimización de thresholds clínicos',
        'Validación en cohortes críticas'
    ]
}

# Añadir modelos ganadores a la configuración
for winner in winners:
    next_phase_config['priority_models'][winner['task']] = {
        'model_name': winner['model'],
        'current_metric': winner['metric'],
        'current_value': winner['value']
    }

# Guardar configuración
import json
with open('../results/phase5b_config.json', 'w') as f:
    json.dump(next_phase_config, f, indent=2)

print("✅ Configuración generada para Fase 5B:")
print("   - Modelos prioritarios identificados")
print("   - Thresholds clínicos definidos")
print("   - Métricas objetivo establecidas")
print(f"   - Archivo: ../results/phase5b_config.json")


## 📝 10. Resumen Ejecutivo

In [ ]:
## 📝 10. Resumen Ejecutivo

# %%
print("\n" + "="*70)
print("📝 RESUMEN EJECUTIVO - FASE 5A COMPLETADA")
print("="*70)

print(f"""
🎯 OBJETIVO CUMPLIDO: Evaluación master de modelos completada

📊 RESULTADOS CLAVE:
   • Total de runs evaluados: {total_runs}
   • Modelos ganadores identificados: {len(winners)}
   • Tareas cubiertas: {', '.join([w['task'] for w in winners])}

🏆 MODELOS SELECCIONADOS:
""")

for winner in winners:
    print(f"   • {winner['task'].upper()}: {winner['model']} ({winner['metric']}: {winner['value']:.4f})")

print(f"""
📁 ARCHIVOS GENERADOS:
   • ../results/model_winners_summary.csv
   • ../results/clinical_optimization_recommendations.csv  
   • ../results/model_evaluation_summary.png
   • ../results/phase5b_config.json

⏭️  SIGUIENTE PASO: Fase 5B - Optimización Clínica
   • Enfoque en modelos de clasificación (prioridad crítica)
   • Calibración de thresholds para uso clínico
   • Análisis de explicabilidad para dashboard

⏱️  TIEMPO ESTIMADO FASE 5B: 45-60 minutos
""")

print("="*70)
print("✅ FASE 5A COMPLETADA - LISTO PARA OPTIMIZACIÓN CLÍNICA")
print("="*70)

---

__Abraham Tartalos__